In [ ]:
# # pip install tslearn
# # %pip install h5py
# !pip install -U tslearn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from scipy.signal import butter, filtfilt

df1 = pd.read_csv('data/cleaned_relax.csv')
df1.columns = ['acc_x', 'acc_y', 'acc_z',
	   'ax_f', 'ay_f', 'az_f', 'vx', 'vy', 'vz', 'px', 'py', 'pz', 'acc_norm',
	   'jerk_x', 'jerk_y', 'jerk_z', 'acc_mean','acc_std']


In [ ]:
df = pd.read_csv('data/relaxes_and_moves_hand.csv')
df[['ax_f', 'ay_f', 'az_f', 'vx', 'vy', 'vz', 'px', 'py', 'pz', 'acc_norm',
	   'jerk_x', 'jerk_y', 'jerk_z', 'acc_mean','acc_std']] = df1[['ax_f', 'ay_f', 'az_f', 'vx', 'vy', 'vz', 'px', 'py', 'pz', 'acc_norm',
	   'jerk_x', 'jerk_y', 'jerk_z', 'acc_mean','acc_std']]

df.head()

In [ ]:
df['sequence_id'].unique()

In [ ]:
X_features = df[['acc_x', 'acc_y', 'acc_z','ax_f', 'ay_f', 'az_f', 'vx', 'vy', 'vz', 'px', 'py', 'pz', 'acc_norm',
       'jerk_x', 'jerk_y', 'jerk_z', 'acc_mean','acc_std']].values

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# NaNを平均値で補完
imputer = SimpleImputer(strategy='mean')
X_features_imputed = imputer.fit_transform(X_features)

# 特徴量の標準化
scaler = StandardScaler()
X_kmeans = scaler.fit_transform(X_features_imputed)

# KMeansクラスタリング（クラスタ数=4）
kmeans = KMeans(n_clusters=4, random_state=42, n_init='auto')
kmeans_labels = kmeans.fit_predict(X_kmeans)

# 結果を df に追加（必要なら）
df['kmeans_cluster'] = kmeans_labels


In [ ]:
import numpy as np
from tslearn.preprocessing import TimeSeriesScalerMeanVariance
from tslearn.clustering import KShape

window_length = 15
step_size = window_length // 2  # 50%オーバーラップ
min_length_for_padding = 20     # これ以上ならパディングで使う

segments = []
segment_ids = []

use_cols = ['acc_x', 'acc_y', 'acc_z','ax_f', 'ay_f', 'az_f', 'vx', 'vy', 'vz', 'px', 'py', 'pz', 'acc_norm',
       'jerk_x', 'jerk_y', 'jerk_z', 'acc_mean','acc_std']

for seq_id, group in df.groupby('sequence_id'):
    X = group[use_cols].values

    seq_len = len(X)

    if seq_len >= window_length:
        for start in range(0, seq_len - window_length + 1, step_size):
            segment = X[start:start+window_length]
            segments.append(segment)
            segment_ids.append(seq_id)
    elif seq_len >= min_length_for_padding:
        # パディング（端の値を繰り返す）
        padded = np.pad(X, ((0, window_length - seq_len), (0, 0)), mode='edge')
        segments.append(padded)
        segment_ids.append(seq_id)
    # else: データが短すぎるので捨てる（3軸×20サンプル未満）

if len(segments) == 0:
    print("Warning: No segments were created.")
else:
    segments = np.array(segments)
    segments_scaled = TimeSeriesScalerMeanVariance().fit_transform(segments)

    kshape = KShape(n_clusters=4, verbose=False, random_state=42)
    kshape_labels = kshape.fit_predict(segments_scaled)

    for i in range(min(10, len(segment_ids))):
        print(f"Segment {i} (sequence_id={segment_ids[i]}) → Cluster {kshape_labels[i]}")


In [ ]:
import matplotlib.pyplot as plt

seq_id = 'SEQ_000007'
df_plot = df[df['sequence_id'] == seq_id]

plt.figure(figsize=(15, 4))
for cluster_id in range(4):
    mask = df_plot['kmeans_cluster'] == cluster_id
    # 'time'列がない場合は'sequence_counter'を横軸に使う
    plt.plot(df_plot['sequence_counter'][mask], df_plot['acc_x'][mask], label=f'Cluster {cluster_id}')

plt.title(f'Sequence {seq_id} - Clustered by KMeans')
plt.xlabel('Time [index]')
plt.ylabel('acc_norm')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

cluster_to_plot = 0
segment_length = 200  # 4秒（50Hz × 4）

if len(segments) == 0 or 'kshape_labels' not in locals():
    print("Warning: No segments or kshape_labels available. Please check your data and clustering step.")
else:
    plt.figure(figsize=(14, 6))
    count = 0

    for i, label in enumerate(kshape_labels):
        if label == cluster_to_plot:
            for d in range(segments.shape[2]):  # 各次元（acc_x, acc_y, acc_z など）を個別に描画
                plt.plot(segments[i, :, d], alpha=0.3, label=f'dim {d}' if count == 0 else "")
            count += 1
        if count >= 20:
            break

    plt.title(f'KShape Cluster {cluster_to_plot} - Sample Segments')
    plt.xlabel('Time step')
    plt.ylabel('Value')
    plt.grid(True)
    plt.legend()
    plt.show()


In [ ]:
# 実際にクラスタリングに使った変数名をここで定義
used_columns = ['acc_x', 'acc_y', 'acc_z']  # or whatever you used

for i, center in enumerate(kshape.cluster_centers_):
    plt.figure(figsize=(10, 5))
    plt.title(f"Cluster {i} Prototype")
    for dim in range(center.shape[1]):
        label = used_columns[dim] if dim < len(used_columns) else f'dim_{dim}'
        plt.plot(center[:, dim], label=label)
    plt.xlabel("Time step")
    plt.ylabel("Standardized Value")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt

target_seq_id = 'SEQ_000220'  # 表示したいシーケンスIDに変更
target_dim = 0  # acc_x=0, acc_y=1, acc_z=2
cluster_colors = ['red', 'blue', 'green', 'orange']

window_length = segments.shape[1]  # セグメント長を安全に取得
time_pointer = 0

plt.figure(figsize=(14, 4))

for seg, seg_id, label in zip(segments, segment_ids, kshape_labels):
    if seg_id == target_seq_id:
        segment = seg[:, target_dim]  # 指定次元の時系列
        color = cluster_colors[label]
        x = range(time_pointer, time_pointer + window_length)
        plt.plot(x, segment, color=color, alpha=0.7)
        time_pointer += window_length

plt.title(f"Sequence {target_seq_id} - Cluster-colored segments (dim={target_dim})")
plt.xlabel("Time step")
plt.ylabel("Standardized Value")
plt.grid(True)
plt.show()


In [ ]:
print(set(segment_ids))